![image.png](https://i.imgur.com/4fN73lZ.png)

## Setup

We install with **uv** and render rollouts as GIFs with `imageio` (consistent with the other RL labs). Walker2d is a **MuJoCo** environment, so we install `gymnasium[mujoco]`.

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[mujoco]" imageio matplotlib torch

# Content

In this lab we implement **DDPG** (Deep Deterministic Policy Gradient) from scratch in PyTorch and train it on the **Walker2d** MuJoCo environment — a continuous-control task where the agent moves leg joints to walk forward.

You can read about Walker2d's observations, actions and rewards [here](https://gymnasium.farama.org/environments/mujoco/walker2d/).

![Walker](https://gymnasium.farama.org/_images/walker2d.gif)

## DDPG

> **Exercise:** This is the student version. Complete the three tasks in `DDPGAgent`: `TASK 1` (the TD target) and `TASK 2` (the actor loss) in `update`, and `TASK 3` (the soft target update) in `soft_update`. Unfinished tasks raise `NotImplementedError`. Hints give the *formula*, not the code. A fully worked version — plus the optional TD3 extension — is in `Day-5_DDPG_Custom_Pytorch_Walker_Solution.ipynb`.

## DDPG: a deterministic actor-critic for continuous control

In a **continuous** action space we cannot take $\arg\max_a Q(s,a)$ — there are infinitely many actions. DDPG's answer is to learn a **deterministic actor** $\mu_\theta(s)$ that outputs the action it believes is best, and train it by pushing it in the direction the critic says is better. This is the **deterministic policy gradient**:

$$\nabla_\theta J \approx \mathbb{E}_s\big[\nabla_a Q_\phi(s,a)\big|_{a=\mu_\theta(s)}\,\nabla_\theta \mu_\theta(s)\big]$$

i.e. maximise $Q_\phi(s, \mu_\theta(s))$ w.r.t. the actor's parameters — in code just `-critic(s, actor(s)).mean()`.

DDPG is **DPG + the DQN tricks**, which is what makes an off-policy actor-critic stable:

1. **Replay buffer** — store past $(s, a, r, s', d)$ transitions and sample mini-batches, breaking correlations and reusing data (off-policy).
2. **Target networks** $\mu_{\theta'}, Q_{\phi'}$ — slow copies used to form the TD target, so the target does not chase the network being trained:
   $$y = r + \gamma\,(1 - d)\,Q_{\phi'}\big(s',\, \mu_{\theta'}(s')\big)$$
   The critic regresses $Q_\phi(s,a)$ onto $y$; the actor then climbs the critic.
3. **Soft target updates** — instead of DQN's hard periodic copy, nudge the targets a little every step: $\theta' \leftarrow \tau\theta + (1-\tau)\theta'$ (small $\tau$, e.g. 0.005).
4. **Exploration noise** — a deterministic policy explores nothing on its own, so we add noise to the action during data collection (Gaussian here; the original paper used temporally-correlated Ornstein–Uhlenbeck noise).

In [ ]:
import collections
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Tiny MLPs + a MuJoCo env step is CPU-bound, so CPU is typically as fast as GPU here.
device = torch.device("cpu")
print("device:", device)

## The model: replay buffer, actor, critic

The **actor** maps a state to an action, squashed with `tanh` and scaled to the action bounds. The **critic** takes the state *and* action and outputs a scalar $Q(s,a)$.

In [ ]:
Experience = collections.namedtuple(
    "Experience", ["state", "action", "reward", "next_state", "done"])


class ExperienceBuffer:
    """A fixed-size replay buffer of past transitions."""
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def append(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        idx = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[i] for i in idx])
        t = lambda x: torch.tensor(np.array(x, dtype=np.float32), device=device)
        return (t(states), t(actions), t(rewards).unsqueeze(1),
                t(next_states), t(dones).unsqueeze(1))


class Actor(nn.Module):
    """Deterministic policy: state -> action in [-action_scale, action_scale]."""
    def __init__(self, state_dim, action_dim, action_scale):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 400)
        self.fc2 = nn.Linear(400, 300)
        self.fc3 = nn.Linear(300, action_dim)
        self.action_scale = action_scale

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.action_scale * torch.tanh(self.fc3(x))


class Critic(nn.Module):
    """Action-value function Q(s, a)."""
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, 400)
        self.fc2 = nn.Linear(400, 300)
        self.fc3 = nn.Linear(300, 1)

    def forward(self, state, action):
        x = F.relu(self.fc1(torch.cat([state, action], dim=1)))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
class DDPGAgent:
    def __init__(self, env, buffer, args):
        self.env = env
        self.buffer = buffer
        self.args = args

        s_dim = env.observation_space.shape[0]
        a_dim = env.action_space.shape[0]
        self.action_scale = float(env.action_space.high[0])

        self.actor = Actor(s_dim, a_dim, self.action_scale).to(device)
        self.actor_target = Actor(s_dim, a_dim, self.action_scale).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())

        self.critic = Critic(s_dim, a_dim).to(device)
        self.critic_target = Critic(s_dim, a_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=args["actor_lr"])
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=args["critic_lr"])

    def select_action(self, state, noise_scale=0.0):
        """Deterministic action + optional Gaussian exploration noise, clipped to bounds."""
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            action = self.actor(s).cpu().numpy()[0]
        if noise_scale > 0:
            action = action + np.random.normal(0, noise_scale * self.action_scale, size=action.shape)
        return action.clip(-self.action_scale, self.action_scale)

    def soft_update(self, net, target_net):
        # TASK 3: the soft target update, in place.
        # HINT: for each (p, tp) in zip(net, target_net): tp = tau*p + (1-tau)*tp
        #       use self.args["tau"] and tp.data.copy_(...).
        raise NotImplementedError("TASK 3: implement the soft target update")

    def update(self):
        """One DDPG gradient step on a sampled mini-batch."""
        states, actions, rewards, next_states, dones = self.buffer.sample(self.args["batch_size"])

        # --- critic update: regress Q(s,a) onto the TD target y ---
        with torch.no_grad():
            # TASK 1: the DDPG TD target using the TARGET actor & critic.
            # HINT: y = r + gamma * (1 - done) * Q_target(s', mu_target(s'))
            #       use self.actor_target, self.critic_target, self.args["gamma"].
            y = None
        if y is None:
            raise NotImplementedError("TASK 1: compute the TD target y")
        critic_loss = F.mse_loss(self.critic(states, actions), y)
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        # --- actor update: climb the critic (the deterministic policy gradient) ---
        # TASK 2: the actor loss (maximise Q(s, mu(s))).
        # HINT: it is the NEGATIVE mean of the critic evaluated at the actor's own action:
        #       -self.critic(states, self.actor(states)).mean().
        actor_loss = None
        if actor_loss is None:
            raise NotImplementedError("TASK 2: compute the actor loss")
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        # --- soft-update both target networks ---
        self.soft_update(self.critic, self.critic_target)
        self.soft_update(self.actor, self.actor_target)

## Initialize the environment and model

In [ ]:
env = gym.make("Walker2d-v5", render_mode="rgb_array")

args = {
    "replay_size": 1_000_000,
    "batch_size": 256,
    "actor_lr": 3e-4,
    "critic_lr": 3e-4,
    "tau": 0.005,
    "gamma": 0.99,
    "exploration_noise": 0.1,
    "start_steps": 10_000,   # collect random transitions before learning starts
}

total_steps = 100_000        # more steps => better walking; try 300k+ for a strong gait

buffer = ExperienceBuffer(args["replay_size"])
agent = DDPGAgent(env, buffer, args)

## Training the model

One gradient step per environment step. We bootstrap on the time-limit truncation but **not** on true termination, so the done mask uses `terminated` only.

**Why the random warm-up first?** For the first `start_steps` we ignore the actor and act **randomly**, only filling the replay buffer — no learning yet. This is a standard DDPG/TD3 *implementation* detail (it is **not** in the lecture slides). The reasoning: a freshly-initialised actor outputs near-noise, so its early actions carry no useful signal, and starting gradient updates on an almost-empty buffer means every mini-batch is tiny and highly correlated — a recipe for the divergence DDPG is already prone to. Collecting a few thousand *diverse* random transitions first gives the critic a broad, decorrelated dataset to fit before the policy starts steering, which noticeably steadies the brittle early phase.

> **Heads-up:** Walker2d is a real MuJoCo control task — training is on the order of ~15 min/run on CPU at `total_steps = 100k`. DDPG is famously *brittle*: expect the reward curve to climb into the hundreds but wobble rather than rise monotonically (the TD3 extension below is the fix). Lower `total_steps` for a quicker pass, or raise it for a stronger gait.

In [ ]:
def train(agent, env, buffer, total_steps, args):
    scores, episode_return = [], 0.0
    state, _ = env.reset(seed=0)
    for step in range(1, total_steps + 1):
        if step < args["start_steps"]:
            action = env.action_space.sample()                    # random warm-up
        else:
            action = agent.select_action(state, args["exploration_noise"])

        next_state, reward, terminated, truncated, _ = env.step(action)
        buffer.append(Experience(state, action, reward, next_state, float(terminated)))
        state = next_state
        episode_return += reward

        if terminated or truncated:
            scores.append(episode_return)
            episode_return = 0.0
            state, _ = env.reset()

        if len(buffer) >= args["start_steps"]:
            agent.update()

        if step % 5000 == 0 and scores:
            print(f"step {step:6d} | last-10 episode return: {np.mean(scores[-10:]):8.1f}")
    env.close()
    return scores

scores = train(agent, env, buffer, total_steps, args)

## Reward curve

In [ ]:
window = 10
smoothed = np.convolve(scores, np.ones(window) / window, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(smoothed)
plt.xlabel("episode")
plt.ylabel(f"episode return ({window}-episode moving avg)")
plt.title("DDPG on Walker2d")
plt.grid(alpha=0.3)
plt.show()

## Visualizing the trained agent

In [ ]:
# Roll out the trained (noise-free) policy and save it as a GIF
import os
os.environ.setdefault("MUJOCO_GL", "egl")   # headless MuJoCo rendering
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)
eval_env = gym.make("Walker2d-v5", render_mode="rgb_array")
state = eval_env.reset()[0]
frames = []
for _ in range(1000):
    frames.append(eval_env.render())
    action = agent.select_action(state)          # no exploration noise at eval
    state, reward, terminated, truncated, _ = eval_env.step(action)
    if terminated or truncated:
        break
eval_env.close()
imageio.mimsave("video/ddpg_walker.gif", frames, fps=30, loop=0)   # loop=0 -> replays forever
display(Image(filename="video/ddpg_walker.gif"))